# Evaluate testing dates: zero-shot results

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay

from smmnlp.evaluation import calculate_binary_metrics, print_metrics
from smmnlp.paths import ProjectPaths
from smmnlp.parsing import expand_partial_date

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Phase 3 results: SMM DxDate Training yes SMM, hx n=77 Testing  yes SMM, hx n=37


## Training & Testing ids


In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_training_true_phase1.xlsx")
true_training = pd.read_excel(readfiletrue)

In [ ]:
readfiletrue = paths.data("Update/True/SMMNLP_testing_true_phase1.xlsx")
true_testing = pd.read_excel(readfiletrue)

In [ ]:
train_ids = true_training["PatientSSN"].unique()
test_ids = true_testing["PatientSSN"].unique()

In [ ]:
len(train_ids)

In [ ]:
len(test_ids)

## Read in true labels


In [ ]:
truedata = paths.data("Update/True/SMMNLP_testing_true_phase3.xlsx")
true_all = pd.read_excel(truedata)

In [ ]:
true_all.head(5)

In [ ]:
len(true_all)

## Read in newly generated results


In [ ]:
readfileresults = paths.data("Update/ExtraLLMRun/phase3_zeroshot_dxdate_human_updtesting.xlsx")
results = pd.read_excel(readfileresults)[
    ["PatientSSN", "NoteDate", "UncertainFlag_LLM", "DxDate_Human", "DxDate_LLM"]
]

In [ ]:
results.head()

In [ ]:
len(results)

## Combine true and result data for performance


In [ ]:
final = true_all.merge(results, on=["PatientSSN", "NoteDate"], how="inner")

In [ ]:
len(final)

In [ ]:
final.head(5)

## uncertainty flag


In [ ]:
conditions = [
    (final["UncertainFlag"] == "yes") & (final["UncertainFlag_LLM"] == True),
    (final["UncertainFlag"] == "no") & (final["UncertainFlag_LLM"] == False),
    (final["UncertainFlag"] == "yes") & (final["UncertainFlag_LLM"] == False),
    (final["UncertainFlag"] == "no") & (final["UncertainFlag_LLM"] == True),
]
choices = ["TP", "TN", "FN", "FP"]
final["UncertainFlag_result"] = np.select(conditions, choices, default="Unknown")

In [ ]:
print(final["UncertainFlag_result"].value_counts())

In [ ]:
final["y_true"] = np.select([final["UncertainFlag"] == "yes"], [1], default=0)
final["y_pred"] = np.select([final["UncertainFlag_LLM"] == True], [1], default=0)

In [ ]:
y_true = final["y_true"]
y_pred = final["y_pred"]
metric_result = calculate_binary_metrics(y_true, y_pred)

In [ ]:
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=metric_result.cm, display_labels=["human review no", "human review yes"]
)
cm_display.plot()
plt.show()

In [ ]:
print_metrics(metric_result)

## SMM Date Extraction Performance


In [ ]:
final["DxDate_Reviewed"] = np.where(
    final["UncertainFlag_LLM"] == True, final["DxDate_Human"], final["DxDate_LLM"]
)

In [ ]:
conditions = [
    (final["DxDate_Reviewed"] != "unknown") & (final["SMM Dx Date"] != "unknown"),
    (final["DxDate_Reviewed"] == "unknown") & (final["SMM Dx Date"] == "unknown"),
    (final["DxDate_Reviewed"] == "unknown") & (final["SMM Dx Date"] != "unknown"),
    (final["DxDate_Reviewed"] != "unknown") & (final["SMM Dx Date"] == "unknown"),
]
choices = ["TP", "TN", "FN", "FP"]
final["Outcome3_result1"] = np.select(conditions, choices, default="Unknown")

In [ ]:
print(final["Outcome3_result1"].value_counts())

In [ ]:
final["y_true"] = np.select([final["SMM Dx Date"] != "unknown"], [1], default=0)
final["y_pred"] = np.select([final["DxDate_Reviewed"] != "unknown"], [1], default=0)

In [ ]:
y_true = final["y_true"]
y_pred = final["y_pred"]
metric_result = calculate_binary_metrics(y_true, y_pred)

In [ ]:
cm_display = ConfusionMatrixDisplay(
    confusion_matrix=metric_result.cm,
    display_labels=["SMM dx date not present", "SMM dx date present"],
)
cm_display.plot()
plt.show()

In [ ]:
print_metrics(metric_result)

In [ ]:
date_diff = final[final["Outcome3_result1"].isin(["TP"])]

In [ ]:
len(date_diff)

In [ ]:
date_diff["DxDate_clean"] = date_diff["DxDate_Reviewed"].apply(expand_partial_date)
date_diff["HxDate"] = date_diff["SMM Dx Date"].apply(expand_partial_date)

In [ ]:
date_diff["True_SMM_date"] = pd.to_datetime(date_diff["HxDate"], format="%m/%d/%Y")
date_diff["LLM_SMM_date"] = pd.to_datetime(date_diff["DxDate_clean"], format="%m/%d/%Y")

In [ ]:
date_diff["date_diff"] = date_diff["LLM_SMM_date"] - date_diff["True_SMM_date"]

In [ ]:
date_diff_days = date_diff["date_diff"].dt.days
plt.figure(figsize=(8, 5))
n, bins, patches = plt.hist(date_diff_days.dropna(), bins=30, color="orange", edgecolor="black")
plt.xlabel("LLM-extracted - Reference standard")
plt.ylabel("Frequency")
plt.title("Histogram of date difference (days) - Zero-shot")
plt.show()

In [ ]:
date_diff_days

In [ ]:
x = date_diff["date_diff"].dt.days.dropna().reset_index(drop=True)
y = np.random.uniform(0, 1, len(x))
fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(x, y, s=320, facecolor="#ED8936", edgecolor="#8B4513", linewidth=1.8)
ax.set_xlabel("Differences in Days - Zero-shot", fontsize=26, color="#8B4513", labelpad=15)
ax.set_yticks([])
ax.set_facecolor("#f3f3f3")
ax.grid(axis="x", color="#d0d0d0", linewidth=1.2)
ax.set_axisbelow(True)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="x", labelsize=20, colors="#8B4513", length=0)
plt.tight_layout()
plt.show()